# History ablation — training diagnostics

Verifies every training run in the 4x2x3x5 grid completed: reached the full epoch budget, wrote a checkpoint, and did not collapse (effective rank stays well above 1). Reads each run's Lightning `metrics.csv`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def _repo_root(start):
    for c in [start.resolve(), *start.resolve().parents]:
        if (c / "plot_style.py").exists() and (c / "experiments").is_dir():
            return c
    return start.resolve()


REPO_ROOT = _repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    from plot_style import FULL_WIDTH, apply_matplotlib_style, palette
    apply_matplotlib_style()
except Exception:
    FULL_WIDTH = 6.5
    palette = {"Dark Blue": "tab:blue", "Dark Red": "tab:red",
               "Light Blue": "#9ecae1", "Light Red": "#fcbba1",
               "Dark Grey": "0.3"}

EXP_DIR = REPO_ROOT / "experiments" / "history_ablation"
MANIFEST = EXP_DIR / "generated_configs" / "manifest.tsv"
ENV_ORDER = ["TwoRoom", "Reacher", "Push-T", "OGBench-Cube"]
METHOD_LABELS = {"idr": "IDR", "sigreg": "SIGReg"}
HISTORIES = [1, 2, 3]


In [ ]:
# Load the grid and locate each run's Lightning CSV log.
grid = pd.read_csv(MANIFEST, sep="\t")
grid["history"] = grid["history"].astype(int)
grid["seed"] = grid["seed"].astype(int)
TRAIN_ROOT = EXP_DIR / "results" / "train"


def metrics_csv(run_name):
    p = TRAIN_ROOT / run_name / "lightning" / "local" / "metrics.csv"
    return p if p.exists() else None


def load_metrics(run_name):
    p = metrics_csv(run_name)
    if p is None:
        return None
    try:
        return pd.read_csv(p)
    except Exception:
        return None


def first_col(df, *substrings):
    for s in substrings:
        hits = [c for c in df.columns if s in c]
        if hits:
            return hits[0]
    return None


print(f"Grid: {len(grid)} runs. CSV logs found: "
      f"{sum(metrics_csv(r) is not None for r in grid['run_name'])}")


In [ ]:
# Completion table: epochs reached, checkpoint present, final losses, effective rank.
def summarize(run):
    run_name = run["run_name"]
    # final_embeddings.pt is written only after training finishes (small; the
    # large last.ckpt need not be transferred for diagnostics).
    done = (TRAIN_ROOT / run_name / "final_embeddings.pt").exists()
    df = load_metrics(run_name)
    row = {
        "run_name": run_name, "env": run["env_label"], "method": run["method"],
        "history": run["history"], "seed": run["seed"],
        "has_final": done, "max_epoch": np.nan,
        "final_train_loss": np.nan, "final_val_loss": np.nan,
        "final_eff_rank": np.nan, "status": "missing",
    }
    if df is None:
        return row
    if "epoch" in df.columns and df["epoch"].notna().any():
        row["max_epoch"] = int(df["epoch"].max())
    tcol = first_col(df, "train/loss", "fit/loss", "train/pred_loss", "fit/pred_loss")
    vcol = first_col(df, "val/pred_loss", "validate/pred_loss", "val/loss")
    ecol = first_col(df, "effective_rank")
    for key, col in [("final_train_loss", tcol), ("final_val_loss", vcol),
                     ("final_eff_rank", ecol)]:
        if col is not None and df[col].notna().any():
            row[key] = float(df[col].dropna().iloc[-1])
    row["status"] = "ok" if done and (row["max_epoch"] or 0) >= 9 else "incomplete"
    return row


completion = pd.DataFrame([summarize(r) for _, r in grid.iterrows()])
n_ok = (completion["status"] == "ok").sum()
print(f"Complete: {n_ok}/{len(completion)}")
if n_ok < len(completion):
    print("\nIncomplete / missing runs:")
    display(completion[completion["status"] != "ok"]
            [["run_name", "has_final", "max_epoch", "status"]])

# Collapse check: flag any run whose final effective rank is suspiciously low.
low_rank = completion[completion["final_eff_rank"] < 2.0].dropna(subset=["final_eff_rank"])
if len(low_rank):
    print("\nPossible collapse (effective rank < 2):")
    display(low_rank[["run_name", "final_eff_rank", "final_val_loss"]])
else:
    print("No collapse flagged (all effective ranks >= 2 where logged).")

display(completion.groupby(["env", "method", "history"])["status"]
        .agg(lambda s: f"{(s=='ok').sum()}/{len(s)}").unstack("history"))


In [ ]:
# Training curves faceted by environment, one line per (method, history), averaged over seeds.
def curve(run_name, ycol_substrings, xcol="epoch"):
    df = load_metrics(run_name)
    if df is None:
        return None, None
    yc = first_col(df, *ycol_substrings)
    if yc is None or xcol not in df.columns:
        return None, None
    sub = df[[xcol, yc]].dropna()
    if sub.empty:
        return None, None
    sub = sub.groupby(xcol)[yc].mean()
    return sub.index.to_numpy(), sub.to_numpy()


def facet(ycol_substrings, title):
    fig, axes = plt.subplots(1, len(ENV_ORDER), figsize=(FULL_WIDTH, 2.4),
                             constrained_layout=True, sharex=True)
    colors = {"idr": palette.get("Dark Red", "tab:red"),
              "sigreg": palette.get("Dark Blue", "tab:blue")}
    styles = {1: "-", 2: "--", 3: ":"}
    for ax, env in zip(axes, ENV_ORDER):
        g = grid[grid["env_label"] == env]
        for method in ["idr", "sigreg"]:
            for h in HISTORIES:
                runs = g[(g["method"] == method) & (g["history"] == h)]["run_name"]
                xs, ys = [], []
                for rn in runs:
                    x, y = curve(rn, ycol_substrings)
                    if x is not None:
                        xs.append(x); ys.append(y)
                if not xs:
                    continue
                n = min(len(x) for x in xs)
                xm = xs[0][:n]
                ym = np.nanmean(np.stack([y[:n] for y in ys]), axis=0)
                ax.plot(xm, ym, styles[h], color=colors[method], lw=1.1,
                        label=f"{METHOD_LABELS[method]} H{h}")
        ax.set_title(env, pad=3)
        ax.set_xlabel("epoch")
        ax.grid(True, lw=0.4)
    axes[0].set_ylabel(title)
    axes[-1].legend(fontsize=5, ncol=2, loc="best")
    plt.show()


facet(["val/pred_loss", "validate/pred_loss", "val/loss"], "val pred loss")
facet(["effective_rank"], "effective rank")
